
# ## Silver Layer - Data Cleaning & Transformation
# Transform and enrich data from Bronze layer


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime

# Setup para Community Edition
catalog = "workspace"
schema = "ecommerce_dq"

# COMMAND ----------

# Cell 1: Load Bronze Tables
print("=" * 60)
print("SILVER LAYER TRANSFORMATION")
print("=" * 60)

bronze_products = spark.table(f"{catalog}.{schema}.bronze_products")
bronze_customers = spark.table(f"{catalog}.{schema}.bronze_customers")
bronze_orders = spark.table(f"{catalog}.{schema}.bronze_orders")
bronze_events = spark.table(f"{catalog}.{schema}.bronze_events")

print("✅ Loaded Bronze tables")


In [0]:

# Cell 2: Transform Products
def transform_silver_products():
    """Clean and enrich products data"""
    df = bronze_products
    
    # Remove duplicates
    df = df.dropDuplicates(subset=["id"])
    
    # Validate price (must be > 0)
    df = df.filter(F.col("price") > 0)
    
    # Standardize category
    df = df.withColumn("category", F.upper(F.col("category")))
    
    # Add data quality flags
    df = df.withColumn("is_valid", F.lit(True))
    df = df.withColumn("quality_check_date", F.current_date())
    df = df.withColumn("dbt_scd_id", F.expr("uuid()"))
    
    # Rename columns
    df = df.withColumnRenamed("title", "product_name")
    df = df.withColumnRenamed("rating", "avg_rating")
    
    # Save to Silver
    table_name = f"{catalog}.{schema}.silver_products"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Silver Products: {df.count()} rows (transformed)")
    return df


In [0]:

# Cell 3: Transform Customers
def transform_silver_customers():
    """Clean and enrich customers data"""
    df = bronze_customers
    
    # Remove duplicates
    df = df.dropDuplicates(subset=["id"])
    
    # Remove nulls in critical fields
    df = df.filter(F.col("email").isNotNull())
    
    # Create full name
    df = df.withColumn("full_name", F.concat_ws(" ", F.col("firstname"), F.col("lastname")))
    
    # Standardize email
    df = df.withColumn("email", F.lower(F.col("email")))
    
    # Add quality flags
    df = df.withColumn("is_valid", F.lit(True))
    df = df.withColumn("quality_check_date", F.current_date())
    
    # Save to Silver
    table_name = f"{catalog}.{schema}.silver_customers"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Silver Customers: {df.count()} rows (transformed)")
    return df


In [0]:

# Cell 4: Transform Orders
def transform_silver_orders():
    """Clean and enrich orders data"""
    df = bronze_orders
    
    # Remove duplicates
    df = df.dropDuplicates(subset=["id"])
    
    # Validate amount (must be > 0)
    df = df.filter(F.col("total_amount") > 0)
    
    # Standardize status
    df = df.withColumn("status", F.lower(F.col("status")))
    
    # Add date fields for analysis
    df = df.withColumn("year", F.year(F.col("order_date")))
    df = df.withColumn("month", F.month(F.col("order_date")))
    df = df.withColumn("day_of_week", F.dayofweek(F.col("order_date")))
    
    # Add quality flags
    df = df.withColumn("is_valid", F.lit(True))
    df = df.withColumn("quality_check_date", F.current_date())
    
    # Save to Silver
    table_name = f"{catalog}.{schema}.silver_orders"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Silver Orders: {df.count()} rows (transformed)")
    return df


In [0]:


# Cell 5: Transform Events
def transform_silver_events():
    """Clean and enrich events data"""
    df = bronze_events
    
    # Remove duplicates
    df = df.dropDuplicates(subset=["event_id"])
    
    # Remove nulls
    df = df.filter(F.col("event_type").isNotNull() & F.col("user_id").isNotNull())
    
    # Extract date fields
    df = df.withColumn("event_date", F.to_date(F.col("timestamp")))
    df = df.withColumn("event_hour", F.hour(F.col("timestamp")))
    
    # Standardize event type
    df = df.withColumn("event_type", F.lower(F.col("event_type")))
    
    # Add quality flags
    df = df.withColumn("is_valid", F.lit(True))
    df = df.withColumn("quality_check_date", F.current_date())
    
    # Save to Silver
    table_name = f"{catalog}.{schema}.silver_events"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Silver Events: {df.count()} rows (transformed)")
    return df

In [0]:

# Cell 6: Run All Transformations
silver_products = transform_silver_products()
silver_customers = transform_silver_customers()
silver_orders = transform_silver_orders()
silver_events = transform_silver_events()

print("\n" + "=" * 60)
print("✅ Silver Layer Complete!")
print("=" * 60)

In [0]:


# COMMAND ----------

# Cell 7: Show Transformations
print("\nSilver Layer Schema (Products):")
silver_products.printSchema()

print("\nSample Data (Products):")
display(silver_products.select("id", "product_name", "category", "price", "is_valid").limit(5))